## Ejercicio 2: Escalamiento de tickets de soporte técnico

### Ficha PEAS

| Elemento                    | Descripción                                                                         |
|-----------------------------|-------------------------------------------------------------------------------------|
| **Percepción (S)**          | Los tres atributos de cada ticket entrante: minutos que lleva esperando, nivel de                                    urgencia declarado ("baja", "media" o "alta") y si el cliente tiene cuenta premium. |
| **Acciones (A)**            | Mantener en nivel 1 / Escalar a nivel 2 / Escalar a nivel 3 (atención inmediata de                                   especialistas).                                                                     |
| **Entorno (E)**             | Una mesa de ayuda con cola de tickets variable, niveles de soporte de capacidad                                      limitada y acuerdos de servicio (SLA) distintos según el tipo de cliente.           |
| **Objetivo**                | Dirigir cada ticket al nivel de soporte adecuado para resolverlo a tiempo, sin                                       saturar a los especialistas con casos que el nivel 1 puede atender.                 |
| **Medida de desempeño (P)** | Porcentaje de tickets resueltos dentro del SLA, tiempo promedio de resolución,                                       satisfacción del cliente y carga de trabajo repartida entre los tres niveles.       |

### Justificación de las reglas

La urgencia alta manda por encima de todo lo demás: si el cliente reporta que su operación está
detenida, hacerlo esperar en cola es el peor resultado posible tanto para él como para la
reputación del servicio. En cambio, la urgencia media sí admite matices, y ahí entra el tiempo de
espera: un ticket que lleva más de una hora sin respuesta ya incumple el compromiso de atención
aunque su severidad sea moderada. La condición premium actúa como acelerador, no como comodín
absoluto, porque esos clientes pagan un SLA más corto, pero escalar automáticamente todos sus
tickets dejaría al nivel 3 sin capacidad para las emergencias reales del resto de usuarios.

**Reglas:**

| Condición                                          | Decisión            |
|----------------------------------------------------|---------------------|
| Urgencia alta (cualquier cliente)                  | Escalar a nivel 3   |
| Urgencia media, espera ≥ 60 min                    | Escalar a nivel 3   |
| Urgencia media, espera ≥ 30 min o cliente premium  | Escalar a nivel 2   |
| Urgencia media, espera corta y cliente regular     | Mantener en nivel 1 |
| Urgencia baja, espera ≥ 120 min                    | Escalar a nivel 2   |
| Urgencia baja, cliente premium con espera ≥ 60 min | Escalar a nivel 2   |
| Urgencia baja, resto de casos                      | Mantener en nivel 1 |

### Ficha PEAS

| Elemento | Descripción |
|---|---|
| **Percepción (S)** | Los tres atributos de cada ticket entrante: minutos que lleva esperando, nivel de urgencia declarado ("baja", "media" o "alta") y si el cliente tiene cuenta premium |
| **Acciones (A)** | Mantener en nivel 1 / Escalar a nivel 2 / Escalar a nivel 3 (atención inmediata de especialistas) |
| **Entorno (E)** | Una mesa de ayuda con cola de tickets variable, niveles de soporte de capacidad limitada y acuerdos de servicio (SLA) distintos según el tipo de cliente |
| **Objetivo** | Dirigir cada ticket al nivel de soporte adecuado para resolverlo a tiempo, sin saturar a los especialistas con casos que el nivel 1 puede atender |
| **Medida de desempeño (P)** | Porcentaje de tickets resueltos dentro del SLA, tiempo promedio de resolución, satisfacción del cliente y carga de trabajo repartida entre los tres niveles |

### Código

In [1]:
def agente_soporte(tiempo_espera_minutos, nivel_urgencia, cliente_premium):
    """
    Agente reactivo simple para escalamiento de tickets de soporte.
    Percepcion (S): tiempo de espera, nivel de urgencia y condicion premium.
    Acciones (A): mantener en nivel 1, escalar a nivel 2, escalar a nivel 3.
    Retorna una tupla (accion, motivo).
    """
    if nivel_urgencia == "alta":
        return "escalar a nivel 3", "urgencia alta reportada por el cliente"

    if nivel_urgencia == "media":
        if tiempo_espera_minutos >= 60:
            return "escalar a nivel 3", f"urgencia media con {tiempo_espera_minutos} min de espera, fuera del SLA"
        if tiempo_espera_minutos >= 30 or cliente_premium:
            motivo = "cliente premium" if cliente_premium else f"{tiempo_espera_minutos} min de espera acumulada"
            return "escalar a nivel 2", f"urgencia media y {motivo}"
        return "mantener en nivel 1", f"urgencia media con espera corta ({tiempo_espera_minutos} min)"

    # Aqui la urgencia es baja
    if tiempo_espera_minutos >= 120:
        return "escalar a nivel 2", f"urgencia baja pero {tiempo_espera_minutos} min de espera excesiva"
    if cliente_premium and tiempo_espera_minutos >= 60:
        return "escalar a nivel 2", f"cliente premium con {tiempo_espera_minutos} min de espera"
    return "mantener en nivel 1", f"urgencia baja sin tiempo de espera critico ({tiempo_espera_minutos} min)"

### Simulación y pruebas

In [2]:
tickets = [
    (5, "alta", False),      # urgencia alta recien llegada -> nivel 3 igual
    (10, "alta", True),      # alta + premium -> nivel 3
    (75, "media", False),    # media pero fuera del SLA -> nivel 3
    (15, "media", True),     # media con poca espera pero premium -> nivel 2
    (10, "media", False),    # media, espera corta, cliente regular -> nivel 1
    (150, "baja", False),    # baja pero espera excesiva -> nivel 2
    (90, "baja", True),      # CONDICIONES EN CONFLICTO: baja + premium + espera alta
    (20, "baja", True),      # baja, premium, espera corta -> nivel 1
]

for espera, urgencia, premium in tickets:
    accion, motivo = agente_soporte(espera, urgencia, premium)
    tipo_cliente = "premium" if premium else "regular"
    print(f"Espera: {espera} min | Urgencia: {urgencia} | Cliente: {tipo_cliente}")
    print(f"   -> {accion.upper()}: {motivo}\n")

Espera: 5 min | Urgencia: alta | Cliente: regular
   -> ESCALAR A NIVEL 3: urgencia alta reportada por el cliente

Espera: 10 min | Urgencia: alta | Cliente: premium
   -> ESCALAR A NIVEL 3: urgencia alta reportada por el cliente

Espera: 75 min | Urgencia: media | Cliente: regular
   -> ESCALAR A NIVEL 3: urgencia media con 75 min de espera, fuera del SLA

Espera: 15 min | Urgencia: media | Cliente: premium
   -> ESCALAR A NIVEL 2: urgencia media y cliente premium

Espera: 10 min | Urgencia: media | Cliente: regular
   -> MANTENER EN NIVEL 1: urgencia media con espera corta (10 min)

Espera: 150 min | Urgencia: baja | Cliente: regular
   -> ESCALAR A NIVEL 2: urgencia baja pero 150 min de espera excesiva

Espera: 90 min | Urgencia: baja | Cliente: premium
   -> ESCALAR A NIVEL 2: cliente premium con 90 min de espera

Espera: 20 min | Urgencia: baja | Cliente: premium
   -> MANTENER EN NIVEL 1: urgencia baja sin tiempo de espera critico (20 min)

